# Data Cleaning — Rossmann Store Sales
Lanjutan dari `Data_Understanding.ipynb`. Sebelum membersihkan data, kita **backup dataset mentah (raw)** terlebih dahulu — baik sebagai file fisik di folder `raw_backup/` maupun sebagai `.copy()` di memori — supaya dataset asli tidak pernah ikut berubah. Semua proses cleaning dilakukan pada **salinan (copy)** dataset, bukan pada `train`/`store` aslinya.


In [1]:
import pandas as pd
import numpy as np
import os
import shutil

## 1. Load Data Mentah (Raw)

In [2]:
train = pd.read_csv('rossmann-store-sales/train.csv')
store = pd.read_csv('rossmann-store-sales/store.csv')

C:\Users\ASUS\AppData\Local\Temp\ipykernel_1944\829133778.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('rossmann-store-sales/train.csv')


## 2. Backup Dataset Mentah (Raw) Sebelum Dibersihkan
Backup fisik file CSV ke folder `raw_backup/`, supaya data mentah asli tetap tersimpan utuh dan bisa dibandingkan/diaudit kapan saja.

In [3]:
raw_backup_dir = 'raw_backup'
os.makedirs(raw_backup_dir, exist_ok=True)

shutil.copy2('rossmann-store-sales/train.csv', f'{raw_backup_dir}/train_raw.csv')
shutil.copy2('rossmann-store-sales/store.csv', f'{raw_backup_dir}/store_raw.csv')

print('Raw files backed up to:', raw_backup_dir)
print(os.listdir(raw_backup_dir))

Raw files backed up to: raw_backup
['store_raw.csv', 'train_raw.csv']


## 3. Buat Salinan (Copy) Dataset untuk Dibersihkan
`train` dan `store` asli **tidak disentuh** mulai dari sini. Semua perubahan hanya dilakukan pada `train_clean` dan `store_clean`.

In [4]:
train_clean = train.copy()
store_clean = store.copy()

print('train_clean shape:', train_clean.shape)
print('store_clean shape:', store_clean.shape)

train_clean shape: (1017209, 9)
store_clean shape: (1115, 10)


## 4. Cleaning `train_clean`

### 4.1 Parse kolom `Date` ke tipe datetime

In [5]:
train_clean['Date'] = pd.to_datetime(train_clean['Date'], format='%Y-%m-%d')
print('Date dtype after parsing:', train_clean['Date'].dtype)
print('Date min:', train_clean['Date'].min())
print('Date max:', train_clean['Date'].max())

Date dtype after parsing: datetime64[us]
Date min: 2013-01-01 00:00:00
Date max: 2015-07-31 00:00:00


### 4.2 Standardisasi `StateHoliday`
Dari data understanding, `StateHoliday` berisi campuran string `'0'` dan angka `0`, plus `'a'`, `'b'`, `'c'`. Kita standarisasi jadi label yang konsisten dan mudah dibaca.

In [6]:
print('Before cleaning:', train_clean['StateHoliday'].unique())

state_holiday_map = {
    0: 'None', '0': 'None',
    'a': 'PublicHoliday',
    'b': 'EasterHoliday',
    'c': 'Christmas'
}
train_clean['StateHoliday'] = train_clean['StateHoliday'].map(state_holiday_map)

print('After cleaning:', train_clean['StateHoliday'].unique())
print(train_clean['StateHoliday'].value_counts())

Before cleaning: ['0' 'a' 'b' 'c' 0]
After cleaning: <StringArray>
['None', 'PublicHoliday', 'EasterHoliday', 'Christmas']
Length: 4, dtype: str
StateHoliday
None             986159
PublicHoliday     20260
EasterHoliday      6690
Christmas          4100
Name: count, dtype: int64


### 4.3 Cast kolom flag (`Open`, `Promo`, `SchoolHoliday`) ke integer 0/1

In [7]:
flag_cols = ['Open', 'Promo', 'SchoolHoliday']
for col in flag_cols:
    train_clean[col] = train_clean[col].astype(int)

print(train_clean[flag_cols].dtypes)

Open             int64
Promo            int64
SchoolHoliday    int64
dtype: object


### 4.4 Tandai hari toko tutup (`is_closed_day`)
Baris dengan `Open = 0` **tidak dihapus** (supaya model forecasting tetap belajar pola hari tutup), tapi diberi flag eksplisit.

In [8]:
train_clean['is_closed_day'] = train_clean['Open'] == 0
print(train_clean['is_closed_day'].value_counts())

is_closed_day
False    844392
True     172817
Name: count, dtype: int64


### 4.5 Cek & hapus duplikat

In [9]:
dup_count = train_clean.duplicated().sum()
print('Duplicated rows before:', dup_count)

train_clean = train_clean.drop_duplicates()
print('train_clean shape after dedup:', train_clean.shape)

Duplicated rows before: 0
train_clean shape after dedup: (1017209, 10)


### 4.6 Validasi ulang `train_clean` setelah cleaning

In [10]:
print('train_clean info:')
print(train_clean.info())
print()
print('train_clean null values:')
print(train_clean.isnull().sum())

train_clean info:
<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 10 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   Store          1017209 non-null  int64         
 1   DayOfWeek      1017209 non-null  int64         
 2   Date           1017209 non-null  datetime64[us]
 3   Sales          1017209 non-null  int64         
 4   Customers      1017209 non-null  int64         
 5   Open           1017209 non-null  int64         
 6   Promo          1017209 non-null  int64         
 7   StateHoliday   1017209 non-null  str           
 8   SchoolHoliday  1017209 non-null  int64         
 9   is_closed_day  1017209 non-null  bool          
dtypes: bool(1), datetime64[us](1), int64(7), str(1)
memory usage: 70.8 MB
None

train_clean null values:
Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
State

## 5. Cleaning `store_clean`

### 5.1 Cek missing value sebelum cleaning

In [11]:
print('Missing values before cleaning:')
print(store_clean.isnull().sum())

Missing values before cleaning:
Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64


### 5.2 Tangani `CompetitionDistance` yang kosong
Diisi dengan nilai maksimum yang ada (mewakili 'kompetitor sangat jauh / tidak diketahui'), dan diberi flag supaya baris hasil imputasi bisa dilacak.

In [12]:
store_clean['competition_distance_imputed'] = store_clean['CompetitionDistance'].isnull()

max_distance = store_clean['CompetitionDistance'].max()
store_clean['CompetitionDistance'] = store_clean['CompetitionDistance'].fillna(max_distance)

print('CompetitionDistance missing after fill:', store_clean['CompetitionDistance'].isnull().sum())
print('Rows imputed:', store_clean['competition_distance_imputed'].sum())

CompetitionDistance missing after fill: 0
Rows imputed: 3


### 5.3 Tangani `CompetitionOpenSinceMonth` / `CompetitionOpenSinceYear`
Kolom ini hanya relevan kalau memang ada data kompetitor. Kita isi 0 (bukan tanggal valid) dan beri flag, daripada menebak tanggal.

In [13]:
store_clean['competition_open_date_missing'] = (
    store_clean['CompetitionOpenSinceMonth'].isnull() |
    store_clean['CompetitionOpenSinceYear'].isnull()
)

store_clean['CompetitionOpenSinceMonth'] = store_clean['CompetitionOpenSinceMonth'].fillna(0).astype(int)
store_clean['CompetitionOpenSinceYear'] = store_clean['CompetitionOpenSinceYear'].fillna(0).astype(int)

print('Flagged rows (missing competition open date):', store_clean['competition_open_date_missing'].sum())

Flagged rows (missing competition open date): 354


### 5.4 Tangani `Promo2SinceWeek` / `Promo2SinceYear` / `PromoInterval`
Kolom ini kosong secara wajar kalau `Promo2 = 0` (toko tidak ikut promo berulang). Kita isi 0 / 'None' dan beri flag khusus untuk kasus yang janggal (Promo2 = 1 tapi datanya kosong).

In [14]:
store_clean['promo2_date_anomaly'] = (
    (store_clean['Promo2'] == 1) &
    (store_clean['Promo2SinceWeek'].isnull() | store_clean['Promo2SinceYear'].isnull())
)

store_clean['Promo2SinceWeek'] = store_clean['Promo2SinceWeek'].fillna(0).astype(int)
store_clean['Promo2SinceYear'] = store_clean['Promo2SinceYear'].fillna(0).astype(int)
store_clean['PromoInterval'] = store_clean['PromoInterval'].fillna('None')

print('Promo2 date anomalies (Promo2=1 but missing dates):', store_clean['promo2_date_anomaly'].sum())
print('PromoInterval unique values:', store_clean['PromoInterval'].unique())

Promo2 date anomalies (Promo2=1 but missing dates): 0
PromoInterval unique values: <StringArray>
['None', 'Jan,Apr,Jul,Oct', 'Feb,May,Aug,Nov', 'Mar,Jun,Sept,Dec']
Length: 4, dtype: str


### 5.5 Cek & hapus duplikat

In [15]:
dup_store = store_clean.duplicated().sum()
print('Duplicated rows before:', dup_store)

store_clean = store_clean.drop_duplicates()
print('store_clean shape after dedup:', store_clean.shape)

Duplicated rows before: 0
store_clean shape after dedup: (1115, 13)


### 5.6 Validasi ulang `store_clean` setelah cleaning

In [16]:
print('store_clean info:')
print(store_clean.info())
print()
print('store_clean null values:')
print(store_clean.isnull().sum())

store_clean info:
<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Store                          1115 non-null   int64  
 1   StoreType                      1115 non-null   str    
 2   Assortment                     1115 non-null   str    
 3   CompetitionDistance            1115 non-null   float64
 4   CompetitionOpenSinceMonth      1115 non-null   int64  
 5   CompetitionOpenSinceYear       1115 non-null   int64  
 6   Promo2                         1115 non-null   int64  
 7   Promo2SinceWeek                1115 non-null   int64  
 8   Promo2SinceYear                1115 non-null   int64  
 9   PromoInterval                  1115 non-null   str    
 10  competition_distance_imputed   1115 non-null   bool   
 11  competition_open_date_missing  1115 non-null   bool   
 12  promo2_date_anomaly            1115 non-n

## 6. Gabungkan `train_clean` + `store_clean`
Sebelum join, kita pastikan dulu tidak ada `Store` di `train_clean` yang tidak punya pasangan di `store_clean` (referential integrity check).

In [17]:
missing_store_keys = set(train_clean['Store']) - set(store_clean['Store'])
print('Store IDs in train_clean but missing from store_clean:', missing_store_keys)

Store IDs in train_clean but missing from store_clean: set()


In [18]:
merged_clean = train_clean.merge(store_clean, on='Store', how='left')

print('merged_clean shape:', merged_clean.shape)
print('merged_clean null values:')
print(merged_clean.isnull().sum())

merged_clean shape: (1017209, 22)
merged_clean null values:
Store                            0
DayOfWeek                        0
Date                             0
Sales                            0
Customers                        0
Open                             0
Promo                            0
StateHoliday                     0
SchoolHoliday                    0
is_closed_day                    0
StoreType                        0
Assortment                       0
CompetitionDistance              0
CompetitionOpenSinceMonth        0
CompetitionOpenSinceYear         0
Promo2                           0
Promo2SinceWeek                  0
Promo2SinceYear                  0
PromoInterval                    0
competition_distance_imputed     0
competition_open_date_missing    0
promo2_date_anomaly              0
dtype: int64


## 7. Simpan Dataset yang Sudah Bersih
Disimpan ke folder terpisah (`cleaned/`), supaya folder data mentah asli (`rossmann-store-sales/`) dan backup-nya (`raw_backup/`) tetap utuh dan tidak tertimpa.

In [19]:
cleaned_dir = 'cleaned'
os.makedirs(cleaned_dir, exist_ok=True)

train_clean.to_csv(f'{cleaned_dir}/train_cleaned.csv', index=False)
store_clean.to_csv(f'{cleaned_dir}/store_cleaned.csv', index=False)
merged_clean.to_csv(f'{cleaned_dir}/merged_cleaned.csv', index=False)

print('Cleaned files saved to:', cleaned_dir)
print(os.listdir(cleaned_dir))

Cleaned files saved to: cleaned
['merged_cleaned.csv', 'store_cleaned.csv', 'train_cleaned.csv']
